# M1 Test Evaluation

M1 pathology-only Cox single-risk model을 이용하여 held-out test split 성능을 평가한다.

주요 출력:
- case-level patient risk score prediction table
- C-index 및 bootstrap 95% CI
- median risk group 기반 Kaplan-Meier curve
- log-rank test
- high-risk vs low-risk hazard ratio

현재 M1 checkpoint는 `cox_partial_likelihood`로 학습된 `n_outputs=1` 모델이므로, horizon별 사망확률/AUROC/AUPRC는 계산하지 않는다.


In [ ]:
# 1. Imports and paths
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import chi2
from torch import nn
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm

try:
    import timm
    from huggingface_hub import hf_hub_download
except ImportError as exc:
    raise ImportError("timm, huggingface_hub가 필요합니다. AGENTS.md의 urban 가상환경에서 실행하세요.") from exc

from scripts.models.discrete_survival import harrell_c_index
from scripts.models.m1_pathology_mil import M1ModelConfig, PathologySpatialMIL, sample_tiles

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_PATH = Path("../../data")
RESULT_PATH = Path("../../results")
MODEL_PATH = Path("../../model")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"

FEATURE_EXTRACTOR_NAME = "UNI2-h"
MODEL_DIR = MODEL_PATH / "pancreatic_cancer_pathology" / "M1" / FEATURE_EXTRACTOR_NAME
BEST_CHECKPOINT_PATH = MODEL_DIR / "best_checkpoint.pt"
TRAIN_CONFIG_PATH = MODEL_DIR / "training_config.json"

RESULT_M1_PATH = RESULT_PATH / "pancreatic_cancer_pathology" / "M1"
LEGACY_M1_PATH = Path("outputs") / "M1"

def first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError("No existing path found: " + "; ".join(str(p) for p in paths))

CASE_SPLIT_PATH = first_existing_path(
    RESULT_M1_PATH / "m1_tcga_cptac_horizon_case_splits.csv",
    LEGACY_M1_PATH / "m1_tcga_cptac_horizon_case_splits.csv",
)
TILE_INDEX_PATH = first_existing_path(
    RESULT_M1_PATH / "m1_tcga_cptac_horizon_tile_index.csv",
    LEGACY_M1_PATH / "m1_tcga_cptac_horizon_tile_index.csv",
)

OUTPUT_PATH = RESULT_PATH / "pancreatic_cancer_pathology" / "M1_test"
TABLE_PATH = OUTPUT_PATH / "tables"
FIGURE_PATH = OUTPUT_PATH / "figures"
for path in [OUTPUT_PATH, TABLE_PATH, FIGURE_PATH]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH, BEST_CHECKPOINT_PATH.exists())
print("TRAIN_CONFIG_PATH:", TRAIN_CONFIG_PATH, TRAIN_CONFIG_PATH.exists())
print("CASE_SPLIT_PATH:", CASE_SPLIT_PATH)
print("TILE_INDEX_PATH:", TILE_INDEX_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)


In [ ]:
# 2. Transform, metrics, and dataset helpers
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)


def get_patch_padding(image_size: int, patch_size: int) -> tuple[int, int, int, int]:
    target_size = int(np.ceil(image_size / patch_size) * patch_size)
    pad_total = max(0, target_size - image_size)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return (pad_left, pad_left, pad_right, pad_right)


def get_model_input_size(image_size: int, patch_size: int) -> int:
    return int(np.ceil(image_size / patch_size) * patch_size)


def get_eval_transform(image_size: int, patch_size: int):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size, patch_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def add_coordinate_columns(tile_df: pd.DataFrame) -> pd.DataFrame:
    tile_df = tile_df.copy()
    if {"x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"}.issubset(tile_df.columns):
        return tile_df
    if {"x_level0", "y_level0"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_level0"].astype(int)
        tile_df["y"] = tile_df["y_level0"].astype(int)
    elif {"x_total_matrix", "y_total_matrix"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_total_matrix"].astype(int)
        tile_df["y"] = tile_df["y_total_matrix"].astype(int)
    else:
        raise ValueError("tile metadata must contain coordinate columns.")
    source_size = tile_df["source_tile_size_px"].astype(float)
    slide_width = int((tile_df["x"].astype(float) + source_size).max())
    slide_height = int((tile_df["y"].astype(float) + source_size).max())
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height
    return tile_df


def bootstrap_ci(df: pd.DataFrame, metric_fn, n_boot: int = 1000, seed: int = 42) -> tuple[float, float, float]:
    point = float(metric_fn(df))
    rng = np.random.default_rng(seed)
    values = []
    n = len(df)
    if n < 2:
        return point, float("nan"), float("nan")
    for _ in range(n_boot):
        sample_idx = rng.integers(0, n, size=n)
        sample_df = df.iloc[sample_idx].reset_index(drop=True)
        try:
            value = float(metric_fn(sample_df))
        except Exception:
            value = float("nan")
        if np.isfinite(value):
            values.append(value)
    if len(values) < 20:
        return point, float("nan"), float("nan")
    low, high = np.percentile(values, [2.5, 97.5])
    return point, float(low), float(high)


def logrank_test(times: np.ndarray, events: np.ndarray, groups: np.ndarray) -> dict[str, float]:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    groups = np.asarray(groups, dtype=int)
    event_times = np.sort(np.unique(times[(events == 1) & np.isfinite(times)]))
    observed_high = 0.0
    expected_high = 0.0
    variance_high = 0.0
    for t in event_times:
        at_risk = times >= t
        event_at_t = (times == t) & (events == 1)
        n = float(at_risk.sum())
        if n <= 1:
            continue
        n_high = float((at_risk & (groups == 1)).sum())
        d = float(event_at_t.sum())
        d_high = float((event_at_t & (groups == 1)).sum())
        expected = d * n_high / n
        var = n_high * (n - n_high) * d * (n - d) / (n * n * max(n - 1.0, 1.0))
        observed_high += d_high
        expected_high += expected
        variance_high += var
    chi2_stat = (observed_high - expected_high) ** 2 / max(variance_high, 1e-12)
    p_value = float(chi2.sf(chi2_stat, df=1))
    return {
        "observed_high": observed_high,
        "expected_high": expected_high,
        "chi2": float(chi2_stat),
        "p_value": p_value,
    }


def cox_binary_hr(times: np.ndarray, events: np.ndarray, group: np.ndarray, max_iter: int = 50) -> dict[str, float]:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=float)
    x = np.asarray(group, dtype=float)
    valid = np.isfinite(times)
    times, events, x = times[valid], events[valid], x[valid]
    if len(np.unique(x)) < 2 or events.sum() == 0:
        return {"hr": float("nan"), "hr_ci_low": float("nan"), "hr_ci_high": float("nan"), "beta": float("nan"), "se": float("nan")}
    beta = 0.0
    info = np.nan
    for _ in range(max_iter):
        score = 0.0
        info = 0.0
        for t, e, xi in zip(times, events, x):
            if e != 1:
                continue
            risk = times >= t
            w = np.exp(np.clip(beta * x[risk], -50, 50))
            sw = w.sum()
            mean_x = (w * x[risk]).sum() / sw
            mean_x2 = (w * x[risk] * x[risk]).sum() / sw
            score += xi - mean_x
            info += mean_x2 - mean_x * mean_x
        if info <= 1e-12:
            break
        step = score / info
        beta += step
        if abs(step) < 1e-6:
            break
    se = float(np.sqrt(1.0 / max(info, 1e-12)))
    return {
        "hr": float(np.exp(beta)),
        "hr_ci_low": float(np.exp(beta - 1.96 * se)),
        "hr_ci_high": float(np.exp(beta + 1.96 * se)),
        "beta": float(beta),
        "se": se,
    }


def km_curve(times: np.ndarray, events: np.ndarray) -> pd.DataFrame:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    order = np.argsort(times)
    times, events = times[order], events[order]
    survival = 1.0
    rows = [{"time": 0.0, "survival": 1.0}]
    for t in np.unique(times[events == 1]):
        at_risk = np.sum(times >= t)
        deaths = np.sum((times == t) & (events == 1))
        if at_risk > 0:
            survival *= 1.0 - deaths / at_risk
            rows.append({"time": float(t), "survival": float(survival)})
    return pd.DataFrame(rows)


class M1EvalDataset(Dataset):
    def __init__(self, case_df: pd.DataFrame, tile_index: pd.DataFrame):
        self.case_df = case_df.reset_index(drop=True).copy()
        self.tile_groups = {
            slide_uid: group.sort_values(["y", "x"] if {"y", "x"}.issubset(group.columns) else ["tile_path"]).reset_index(drop=True)
            for slide_uid, group in tile_index.groupby("slide_uid")
        }

    def __len__(self):
        return len(self.case_df)

    def __getitem__(self, idx):
        row = self.case_df.iloc[idx]
        tiles = self.tile_groups[row["slide_uid"]]
        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "slide_uid": row["slide_uid"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "os_time_days": float(row["os_time_days"]),
            "os_event": int(row["os_event"]),
        }


In [ ]:
# 3. Load trained Cox checkpoint and UNI/UNI2-h feature extractor
with open(TRAIN_CONFIG_PATH, "r", encoding="utf-8") as f:
    train_config = json.load(f)
print(json.dumps(train_config, indent=2, ensure_ascii=False))

PATCH_INPUT_SIZE = int(train_config.get("patch_input_size", 512))
MAX_TILES_PER_SLIDE = int(train_config.get("max_tiles_per_slide", 512))
FEATURE_BATCH_SIZE = int(train_config.get("feature_batch_size", 64))
SPATIAL_DIM = int(train_config.get("spatial_dim", 128))
FUSION_DIM = int(train_config.get("fusion_dim", 128))
MIL_HIDDEN_DIM = int(train_config.get("mil_hidden_dim", 64))
DROPOUT = float(train_config.get("dropout", 0.5))
USE_SPATIAL_EMBEDDING = bool(train_config.get("use_spatial_embedding", False))
POOLING_MODE = str(train_config.get("pooling_mode", "risk_topk"))
FEATURE_EXTRACTOR_NAME = str(train_config.get("feature_extractor", "UNI2-h"))
N_OUTPUTS = 1

UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}

FEATURE_EXTRACTOR_PATCH_SIZE = int(UNI_BACKBONES[FEATURE_EXTRACTOR_NAME]["timm_kwargs"]["patch_size"])
MODEL_INPUT_SIZE = get_model_input_size(PATCH_INPUT_SIZE, FEATURE_EXTRACTOR_PATCH_SIZE)
PATCH_PADDING = get_patch_padding(PATCH_INPUT_SIZE, FEATURE_EXTRACTOR_PATCH_SIZE)
eval_transform = get_eval_transform(PATCH_INPUT_SIZE, FEATURE_EXTRACTOR_PATCH_SIZE)


def load_uni_feature_extractor(name: str, device: torch.device) -> tuple[nn.Module, int]:
    if name not in UNI_BACKBONES:
        raise ValueError(f"Unsupported feature extractor: {name}")
    cfg = UNI_BACKBONES[name]
    print(f"Loading {name}: {cfg['repo_id']} / {cfg['timm_kwargs']['model_name']}")
    model = timm.create_model(pretrained=False, **cfg["timm_kwargs"])
    weight_path = hf_hub_download(repo_id=cfg["repo_id"], filename=cfg["filename"], local_files_only=False)
    state_dict = torch.load(weight_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("load_state_dict warning", {"missing": len(missing), "unexpected": len(unexpected)})
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device)
    feature_dim = int(getattr(model, "num_features", cfg["feature_dim"]))
    print(f"{name} loaded. feature_dim={feature_dim}")
    return model, feature_dim


feature_extractor, M1_FEATURE_DIM = load_uni_feature_extractor(FEATURE_EXTRACTOR_NAME, DEVICE)
model = PathologySpatialMIL(
    feature_extractor=feature_extractor,
    config=M1ModelConfig(
        feature_dim=M1_FEATURE_DIM,
        coord_dim=6,
        spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
        use_spatial_embedding=USE_SPATIAL_EMBEDDING,
        pooling_mode=POOLING_MODE,
    ),
).to(DEVICE)

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location="cpu")
missing, unexpected = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.eval()
print("checkpoint epoch:", checkpoint.get("epoch"))
print("missing:", missing)
print("unexpected:", unexpected)
print("MODEL_INPUT_SIZE:", MODEL_INPUT_SIZE, "PATCH_PADDING:", PATCH_PADDING)
print("N_OUTPUTS:", N_OUTPUTS, "Cox patient-level risk score")


In [ ]:
# 4. Build held-out evaluation datasets
split_df = pd.read_csv(CASE_SPLIT_PATH)
tile_index_df = pd.read_csv(TILE_INDEX_PATH)

if "slide_uid" not in split_df.columns:
    split_df["slide_uid"] = split_df["dataset"].astype(str) + "::" + split_df["case_id"].astype(str)
if "slide_uid" not in tile_index_df.columns:
    tile_index_df["slide_uid"] = tile_index_df["dataset"].astype(str) + "::" + tile_index_df["case_id"].astype(str)

tile_index_df = add_coordinate_columns(tile_index_df)
tile_index_df["tile_path"] = tile_index_df["tile_path"].astype(str)
tile_index_df = tile_index_df[tile_index_df["tile_path"].map(lambda p: Path(p).exists())].copy()

test_df = split_df[split_df["split"].eq("test")].copy()
test_df["os_time_days"] = pd.to_numeric(test_df["os_time_days"], errors="coerce")
test_df["os_event"] = pd.to_numeric(test_df["os_event"], errors="coerce")
test_df = test_df[test_df["os_time_days"].notna() & test_df["os_event"].isin([0, 1])].copy()

# Combined training을 사용했으므로 CPTAC는 external이 아니라 held-out CPTAC test subset으로 해석합니다.
eval_cohorts = {
    "combined_test": test_df,
    "tcga_test": test_df[test_df["dataset"].eq("TCGA_PAAD")].copy(),
    "cptac_test": test_df[test_df["dataset"].eq("CPTAC_PDAC")].copy(),
}

eval_datasets = {}
for name, cohort_df in eval_cohorts.items():
    cohort_tile_index = tile_index_df[tile_index_df["slide_uid"].isin(cohort_df["slide_uid"])].copy()
    eval_datasets[name] = M1EvalDataset(cohort_df, cohort_tile_index)
    print(name, "cases:", len(cohort_df), "tiles:", len(cohort_tile_index), "events:", int(cohort_df["os_event"].sum()) if len(cohort_df) else 0)

display(pd.crosstab(test_df["dataset"], test_df["os_event"]))


In [ ]:
# 5. Run Cox risk inference
def load_tile_tensor_batch(tile_paths: list[str]) -> torch.Tensor:
    images = []
    for path in tile_paths:
        with Image.open(path) as image_file:
            image = image_file.convert("RGB")
        images.append(eval_transform(image))
    return torch.stack(images, dim=0).to(DEVICE, non_blocking=True)


@torch.no_grad()
def predict_dataset(dataset: M1EvalDataset, dataset_name: str) -> pd.DataFrame:
    rows = []
    for sample in tqdm(dataset, desc=f"Predict {dataset_name}", unit="case"):
        selected_paths, selected_coords, _ = sample_tiles(
            sample["tile_paths"],
            sample["coords"],
            max_tiles=MAX_TILES_PER_SLIDE,
            training=False,
        )
        tile_images = load_tile_tensor_batch(selected_paths)
        coords = selected_coords.to(DEVICE, non_blocking=True)
        outputs = model(tile_images, coords)
        risk_score = float(outputs["logits"].reshape(-1).detach().cpu().item())
        rows.append({
            "eval_dataset": dataset_name,
            "dataset": sample["dataset"],
            "case_id": sample["case_id"],
            "slide_uid": sample["slide_uid"],
            "os_time_days": float(sample["os_time_days"]),
            "os_event": int(sample["os_event"]),
            "risk_score": risk_score,
            "n_tiles_used": len(selected_paths),
            "n_tiles_total": len(sample["tile_paths"]),
        })
        del tile_images, coords, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    pred_df = pd.DataFrame(rows)
    if not pred_df.empty:
        pred_df["risk_percentile"] = pred_df["risk_score"].rank(pct=True)
    return pred_df

pred_tables = {name: predict_dataset(ds, name) for name, ds in eval_datasets.items()}
for name, pred_df in pred_tables.items():
    out_path = OUTPUT_PATH / f"m1_{name}_cox_predictions.csv"
    pred_df.to_csv(out_path, index=False)
    print(name, pred_df.shape, "saved:", out_path)
    display(pred_df.head())

all_pred_df = pd.concat(pred_tables.values(), ignore_index=True)
all_pred_df.to_csv(OUTPUT_PATH / "m1_all_test_cox_predictions.csv", index=False)


In [ ]:
# 6. Metrics, bootstrap CI, log-rank test, HR, and KM curves
def evaluate_predictions(pred_df: pd.DataFrame, name: str, n_boot: int = 1000) -> dict:
    metrics = {
        "dataset": name,
        "n_cases": int(len(pred_df)),
        "n_events": int(pred_df["os_event"].sum()) if len(pred_df) else 0,
    }
    if len(pred_df) < 3 or pred_df["os_event"].nunique() < 2:
        metrics.update({
            "c_index": float("nan"), "c_index_ci_low": float("nan"), "c_index_ci_high": float("nan"),
            "risk_cutoff_median": float("nan"), "logrank_p_value": float("nan"), "logrank_chi2": float("nan"),
            "hr_high_vs_low": float("nan"), "hr_ci_low": float("nan"), "hr_ci_high": float("nan"),
        })
        return metrics

    c_point, c_low, c_high = bootstrap_ci(
        pred_df,
        lambda d: harrell_c_index(d["os_time_days"].to_numpy(float), d["os_event"].to_numpy(int), d["risk_score"].to_numpy(float)),
        n_boot=n_boot,
        seed=SEED,
    )
    cutoff = float(pred_df["risk_score"].median())
    group = (pred_df["risk_score"].to_numpy(float) >= cutoff).astype(int)
    lr = logrank_test(pred_df["os_time_days"].to_numpy(float), pred_df["os_event"].to_numpy(int), group)
    hr = cox_binary_hr(pred_df["os_time_days"].to_numpy(float), pred_df["os_event"].to_numpy(int), group)
    metrics.update({
        "c_index": c_point,
        "c_index_ci_low": c_low,
        "c_index_ci_high": c_high,
        "risk_cutoff_median": cutoff,
        "logrank_p_value": lr["p_value"],
        "logrank_chi2": lr["chi2"],
        "hr_high_vs_low": hr["hr"],
        "hr_ci_low": hr["hr_ci_low"],
        "hr_ci_high": hr["hr_ci_high"],
    })
    return metrics


def assign_risk_group(pred_df: pd.DataFrame) -> pd.DataFrame:
    pred_df = pred_df.copy()
    if pred_df.empty:
        pred_df["risk_group"] = []
        return pred_df
    cutoff = float(pred_df["risk_score"].median())
    pred_df["risk_group"] = np.where(pred_df["risk_score"] >= cutoff, "High risk", "Low risk")
    return pred_df


def plot_km(pred_df: pd.DataFrame, title: str, path: Path) -> None:
    pred_df = assign_risk_group(pred_df)
    if pred_df.empty or pred_df["risk_group"].nunique() < 2:
        return
    plt.figure(figsize=(6, 5))
    for group_name, part in pred_df.groupby("risk_group"):
        km = km_curve(part["os_time_days"].to_numpy(float), part["os_event"].to_numpy(int))
        plt.step(km["time"] / 30.4375, km["survival"], where="post", label=f"{group_name} (n={len(part)})")
    plt.title(title)
    plt.xlabel("Months")
    plt.ylabel("Overall survival probability")
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

metric_rows = []
for name, pred_df in pred_tables.items():
    metrics = evaluate_predictions(pred_df, name=name, n_boot=1000)
    metric_rows.append(metrics)
    pred_with_group = assign_risk_group(pred_df)
    pred_with_group.to_csv(OUTPUT_PATH / f"m1_{name}_cox_predictions_with_risk_group.csv", index=False)
    plot_km(
        pred_with_group,
        title=f"M1 {name} KM by median Cox risk\nC-index={metrics['c_index']:.3f}, log-rank p={metrics['logrank_p_value']:.3g}",
        path=FIGURE_PATH / f"m1_{name}_km_curve.png",
    )

metric_df = pd.DataFrame(metric_rows)
metric_df.to_csv(OUTPUT_PATH / "m1_test_cox_metric_summary.csv", index=False)
display(metric_df)


In [ ]:
# 7. Publication-ready tables and short summary
def format_ci(point: float, low: float, high: float, digits: int = 3) -> str:
    if not np.isfinite(point):
        return "NA"
    if not np.isfinite(low) or not np.isfinite(high):
        return f"{point:.{digits}f}"
    return f"{point:.{digits}f} ({low:.{digits}f}-{high:.{digits}f})"

summary_table = metric_df.copy()
summary_table["C-index (95% CI)"] = summary_table.apply(
    lambda r: format_ci(r["c_index"], r["c_index_ci_low"], r["c_index_ci_high"]), axis=1
)
summary_table["HR high vs low (95% CI)"] = summary_table.apply(
    lambda r: format_ci(r["hr_high_vs_low"], r["hr_ci_low"], r["hr_ci_high"], digits=2), axis=1
)
summary_table["Log-rank p"] = summary_table["logrank_p_value"].map(
    lambda x: "NA" if not np.isfinite(x) else f"{x:.3g}"
)
summary_table = summary_table[[
    "dataset",
    "n_cases",
    "n_events",
    "C-index (95% CI)",
    "HR high vs low (95% CI)",
    "Log-rank p",
]]
summary_table.to_csv(TABLE_PATH / "m1_cox_performance_table.csv", index=False)
display(summary_table)

# Risk score distribution figure
plt.figure(figsize=(7, 4))
for name, pred_df in pred_tables.items():
    if not pred_df.empty:
        plt.hist(pred_df["risk_score"], bins=15, alpha=0.45, label=name)
plt.xlabel("Cox risk score")
plt.ylabel("Number of cases")
plt.title("M1 Cox risk score distribution")
plt.legend()
plt.grid(alpha=0.25)
plt.savefig(FIGURE_PATH / "m1_cox_risk_score_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

summary_lines = []
for _, row in metric_df.iterrows():
    summary_lines.append(
        f"{row['dataset']}: n={int(row['n_cases'])}, events={int(row['n_events'])}, "
        f"C-index={row['c_index']:.3f} (95% CI {row['c_index_ci_low']:.3f}-{row['c_index_ci_high']:.3f}), "
        f"HR={row['hr_high_vs_low']:.2f} (95% CI {row['hr_ci_low']:.2f}-{row['hr_ci_high']:.2f}), "
        f"log-rank p={row['logrank_p_value']:.3g}."
    )
for line in summary_lines:
    print(line)

with open(OUTPUT_PATH / "m1_test_cox_summary_text.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines) + "\n")
